# Groundtruth: NAIP county precompute on a Colab GPU

Embeds every 224 px chip of Mecklenburg County for one NAIP year, using the **locked v3 format** (`docs/PRECOMPUTE.md`):
DINOv2 8×8 cells projected to 192-d (fixed PCA basis from the repo) and whole-chip OpenCLIP, both fp16.

* **No keys, no logins, no Google data.** Imagery is public USDA NAIP, read anonymously from Microsoft Planetary Computer. Code comes from the public GitHub repo.
* Everything is written to this Colab VM's disk (`/content`). Nothing goes to Drive.
* **Resumable:** each NAIP scene becomes one shard. If a cell stops, run it again and it skips finished scenes. If the *runtime* resets, the disk is wiped. Download the parts you already have first (step 6 can run at any time).

**Steps:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom. Step 5 takes about 30–60 min.
Step 6 packs the shards into ~900 MB zip parts with a `MANIFEST.json`. Download **all parts + `MANIFEST.json` + `MANIFEST.sha256`**.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
YEAR = 2022          # NAIP year to embed
TAG = "precompute-v3"  # repo tag = locked format + PCA basis
REPO = "https://github.com/sensordyme-debug/groundtruth"

In [ ]:
# 2. Dependencies (pinned; MIT/BSD/Apache only). Colab's preinstalled CUDA torch is kept.
!pip -q install "open_clip_torch==4.2.1" "transformers==5.22.2" "faiss-cpu==1.15.1" "rasterio==1.5.2" \
    "pystac-client==0.9.1" "planetary-computer==1.0.0" "httpx==0.28.1" "pydantic==2.14.3" \
    "shapely==2.2.0" "pyproj==3.8.0" "pyarrow==26.0.1"
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# 3. Code at the locked tag
import os
if not os.path.exists("/content/groundtruth"):
    !git clone -q --branch $TAG --depth 1 $REPO /content/groundtruth
%cd /content/groundtruth
!git log -1 --format='%h %s'
os.environ.update(GROUNDTRUTH_DATA="/content/gt_data", GROUNDTRUTH_COG_CACHE="/content/gt_cog",
                  GROUNDTRUTH_DEVICE="cuda", PYTHONPATH="/content/groundtruth/engine:/content/groundtruth")

In [ ]:
# 4. County boundary (US Census) + NAIP scene list (Planetary Computer STAC), then the plan
!python scripts/fetch_boundary_naip.py
!python scripts/precompute_county.py --aoi county --year $YEAR --plan-only 2>&1 | grep plan:

In [ ]:
# 5. Embed the county (resumable: re-run this cell to continue after any interruption)
!python scripts/precompute_county.py --aoi county --year $YEAR --batch 128 --no-index 2>&1 | grep -E "plan:|device=|scene [0-9]|done:|WARNING|ERROR"

In [ ]:
# 6. Pack shards into ~900 MB parts + SHA-256 manifest (safe to run repeatedly; only new shards are added)
import glob
base = glob.glob(f"/content/gt_data/precompute/*/county/{YEAR}")[0]
!python scripts/pack_shards.py {base}/shards /content/gt_out --part-mb 900 --meta {base}/plan.json {base}/precompute.log
!ls -la /content/gt_out

In [ ]:
# 7. Download everything in /content/gt_out (one browser download per file).
#    Alternatively use the Files panel on the left: right-click each file → Download.
from google.colab import files
import os
for f in sorted(os.listdir("/content/gt_out")):
    files.download(os.path.join("/content/gt_out", f))

## Back on the Groundtruth PC
Put every downloaded file into `data/incoming/2022/` in the repo. Then (in WSL):
```bash
bash scripts/gt.sh python scripts/verify_shards.py data/incoming/2022 --year 2022
```
It checks each part's SHA-256 against `MANIFEST.json`, then every shard's hash and vector widths. It unpacks into
`data/precompute/<version>/county/2022/shards/` and builds the index. If anything mismatches, it stops and indexes nothing.